# LMCBert reproduction — v6: seed-replicated comparison, TF-IDF + Indegree vs MiniLM + Indegree

Same pipeline as v3. This notebook runs ONE controlled experiment: two graph summarisers that differ only in the edge similarity
(lexical TF-IDF cosine vs MiniLM sentence-embedding cosine), both ranked by weighted Indegree, each trained with 3 random seeds
under identical settings, then reports accuracy / F1 / AUC as mean ± SD and significance tests (Welch's t-test across seeds,
McNemar and a paired bootstrap on the 1,000 test papers).

(v5 fixes a syntax error in cell 3b.) Setup on Drive: put `lmcbert_code.zip` in `MyDrive/lmcbert_colab/`. Everything is cached/resumable as before.

In [ ]:
#@title 1. Mount Drive, check GPU
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
ROOT = '/content/drive/MyDrive/lmcbert_colab'
os.makedirs(ROOT, exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout or 'NO GPU - change runtime type to T4')
assert os.path.exists(f'{ROOT}/lmcbert_code.zip'), 'Put lmcbert_code.zip in MyDrive/lmcbert_colab/ first'

In [ ]:
#@title 2. Install deps, unpack code, link runs/ to Drive
%pip -q install ijson gdown sentence-transformers
import shutil, os
if os.path.exists('/content/lmcbert_repro'): shutil.rmtree('/content/lmcbert_repro')
os.makedirs('/content/lmcbert_repro'); os.chdir('/content/lmcbert_repro')
!unzip -q -o "$ROOT/lmcbert_code.zip" -d /content/lmcbert_repro
os.makedirs(f'{ROOT}/runs_full', exist_ok=True)
if os.path.islink('runs') or os.path.exists('runs'): os.remove('runs') if os.path.islink('runs') else shutil.rmtree('runs')
os.symlink(f'{ROOT}/runs_full', 'runs')
!ls
!grep -c "preds.json" train.py
import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
#@title 3. Base data (cached on Drive from v3; rebuilt only if missing)
DATA = f'{ROOT}/data/splits'
assert all(os.path.exists(f'{DATA}/{s}.jsonl') for s in ('train','val','test')), 'base splits missing - run v3 cell 3 first'
!wc -l "$DATA"/*.jsonl

In [ ]:
#@title 3b. Graph summaries for BOTH similarities (Indegree). embed is cached from v3; tfidf is built here (~1 h on CPU)
CENTRALITY = "indegree"
GD = {}
for SIM in ("embed", "tfidf"):
    G = f'{ROOT}/data/splits_graph_{SIM}_{CENTRALITY}'; GD[SIM] = G
    os.makedirs(G, exist_ok=True)
    if all(os.path.exists(f'{G}/{s}.sum.jsonl') for s in ('train','val','test')):
        print(SIM, ': graph summaries already on Drive - skipping'); continue
    for s in ('val','test','train'):
        if not os.path.exists(f'{G}/{s}.sum.jsonl'):
            !python -u summarize.py --inp "$DATA/{s}.jsonl" --out "$G/{s}.sum.jsonl.part" --backend graph --sim $SIM --centrality $CENTRALITY --resume
            !mv "$G/{s}.sum.jsonl.part" "$G/{s}.sum.jsonl"
for SIM, G in GD.items():
    print('==', SIM)
    !wc -l "$G"/*.sum.jsonl

In [ ]:
#@title 4. Seed-replicated training: 2 summarisers x 5 seeds, identical settings (~1 h per run on a T4; resumable)
SEEDS = [42, 43, 44, 45, 46]   # 42-44 already done on Drive and skipped; 45 and 46 are new
MAX_LEN, EPOCHS, PATIENCE = 512, 10, 3
COMMON = f'--input summary --contrastive none --epochs {EPOCHS} --patience {PATIENCE} --batch 16 --accum 2 --max_len {MAX_LEN} --lr 2e-5 --fp16 --resume --skip_done --clean --progress 200'
for SIM in ("tfidf", "embed"):
    G = GD[SIM]
    for seed in SEEDS:
        name = f'graph_{SIM}_indegree_bert_summary_s{seed}'
        print(f'\n=================== {name} ===================', flush=True)
        !python -u train.py --data "$G" $COMMON --seed $seed --name $name
!python collect_results.py

In [ ]:
#@title 5. Statistics: mean ± SD over seeds, Welch t-test, McNemar, paired bootstrap
!python stats_compare.py --a "runs/graph_tfidf_indegree_bert_summary_s*" --b "runs/graph_embed_indegree_bert_summary_s*" --name_a "TF-IDF + Indegree" --name_b "MiniLM + Indegree"
!cp runs/stats_*.md runs/results.md runs/results.csv "$ROOT/"
print('stats copied to', ROOT)